In [10]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.inspection import permutation_importance

from scipy import stats
import matplotlib.pyplot as plt

In [8]:
data_folder = 'C:\\Users\\shirl\\Documents\\Studie\\2025-2026\\Thesis\\personalized-coping-challenges\\data\\'
save_folder = 'functions\\2'
filename = 'simulation_results_2.csv'
df = pd.read_csv(data_folder + filename)

action_file = 'normalized_challenges.csv'
action_df = pd.read_csv(data_folder + action_file)


expert_score_cols = ['score_acceptance', 'score_distraction', 'score_problem_solving', 'score_social_support']
expert_score_matrix = action_df[expert_score_cols].values

category_mapping = {"acceptance": 0, "distraction": 1, "problem_solving": 2, "social_support": 3}

action_df['category_id'] = action_df['category'].map(category_mapping)
action_categories = action_df['category_id'].values

df = df.merge(action_df[['action_id', 'category_id'] + expert_score_cols], on='action_id', how='left')

NUM_ACTIONS = len(action_df)

# possible state features
state_features = ['tiredness', 'time_avail', 'pu_state', 'rw_state', 'motivation']
reward_signals = ['obs_time', 'obs_liked', 'obs_pu']

df['s_next'] = df.groupby('user_id')[state_features].shift(-1).values.tolist()
df['s_current'] = df[state_features].values.tolist()

# drop rows where s_next contains NaN values (last row of each user)
df = df[~df['s_next'].apply(lambda x: any(pd.isna(i) for i in x))]
df = df.drop(state_features, axis=1)

# reward for skill improvement = reward for completing the action * (sum of skill tiers after - sum of skill tiers before) / max possible increase in skill tiers
df['r_expert'] = df[expert_score_cols].sum(axis=1) / 4  # alternative reward based on expert scores for the action
# df['r_diversity'] = df['completed'] * (1 - (1 / (MAX_COUNT + 1)) * df.apply(lambda row: row['s_count'][action_categories[row['action_id']]], axis=1))
df[reward_signals] = df[reward_signals].div(7)  # Normalize rewards to [0, 1] range

df = df[['s_current', 'action_id', 's_next', 'r_expert', 'obs_time', 'obs_liked', 'obs_pu', 'obs_diff']].fillna(0)

state_cols = [f"s{i}" for i in range(5)]

df[state_cols] = pd.DataFrame(df["s_current"].tolist(), index=df.index)

next_state_cols = [f"s_next{i}" for i in range(5)]
df[next_state_cols] = pd.DataFrame(df["s_next"].tolist(), index=df.index)

df.head()

,s_current,action_id,s_next,r_expert,obs_time,obs_liked,obs_pu,obs_diff,s0,s1,s2,s3,s4,s_next0,s_next1,s_next2,s_next3,s_next4
0,"[3.0, 3.0, 4.0, 1.0, 4.0]",44,"[5.0, 3.0, 5.0, 1.0, 5.0]",0.333333,0.714286,0.571429,0.714286,3.0,3.0,3.0,4.0,1.0,4.0,5.0,3.0,5.0,1.0,5.0
1,"[5.0, 3.0, 5.0, 1.0, 5.0]",92,"[4.0, 4.0, 5.0, 1.0, 5.0]",0.166667,0.000000,0.000000,0.000000,0.0,5.0,3.0,5.0,1.0,5.0,4.0,4.0,5.0,1.0,5.0
2,"[4.0, 4.0, 5.0, 1.0, 5.0]",101,"[4.0, 4.0, 6.0, 2.0, 6.0]",0.250000,0.428571,1.000000,1.000000,6.0,4.0,4.0,5.0,1.0,5.0,4.0,4.0,6.0,2.0,6.0
3,"[4.0, 4.0, 6.0, 2.0, 6.0]",46,"[3.0, 5.0, 7.0, 2.0, 7.0]",0.250000,0.571429,0.571429,1.000000,4.0,4.0,4.0,6.0,2.0,6.0,3.0,5.0,7.0,2.0,7.0
4,"[3.0, 5.0, 7.0, 2.0, 7.0]",79,"[5.0, 4.0, 7.0, 2.0, 7.0]",0.250000,0.000000,0.000000,0.000000,0.0,3.0,5.0,7.0,2.0,7.0,5.0,4.0,7.0,2.0,7.0


In [ ]:


FEATURE_NAMES = ['u1', 'u2', 'u3', 'C_AC', 'C_DI', 'C_PS', 'C_SS']  # adjust to your state features

# ── helper ──────────────────────────────────────────────────────────────────

def prepare_inputs(df):
    """Concatenate state + action into a single feature matrix."""
    states = np.stack(df['state'].values)
    actions = df['action'].values.reshape(-1, 1)
    return np.hstack([states, actions])

# ── 1. Random Forest feature importance ─────────────────────────────────────

def rf_feature_importance(df, plot=True):
    X = prepare_inputs(df)
    next_states = np.stack(df['next_state'].values)
    rewards = df['rewards'].values

    input_names = FEATURE_NAMES + ['action']

    # --- next state (multi-output classifier, one tree per state dimension) ---
    rf_transition = MultiOutputClassifier(RandomForestClassifier(n_estimators=100, random_state=42))
    rf_transition.fit(X, next_states)
    # average importance across all output dimensions
    transition_importance = np.mean(
        [est.feature_importances_ for est in rf_transition.estimators_], axis=0
    )

    # --- reward (regressor) ---
    rf_reward = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_reward.fit(X, rewards)
    reward_importance = rf_reward.feature_importances_

    results = pd.DataFrame({
        'feature': input_names,
        'transition_importance': transition_importance,
        'reward_importance': reward_importance,
        'mean_importance': (transition_importance + reward_importance) / 2
    }).sort_values('mean_importance', ascending=False)

    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        for ax, col, title in zip(axes,
                                   ['transition_importance', 'reward_importance', 'mean_importance'],
                                   ['Transition model', 'Reward model', 'Mean']):
            ax.barh(results['feature'], results[col])
            ax.set_title(title)
            ax.set_xlabel('Importance')
        plt.tight_layout()
        plt.show()

    return results

# ── 2. G-algorithm ───────────────────────────────────────────────────────────

class GAlgorithm:
    """
    Offline adaptation of the G algorithm (Chapman & Kaelbling, 1991).
    Recursively splits the state space on the most statistically relevant
    feature using a t-test on reward distributions.
    """

    def __init__(self, feature_names=None, min_samples=30, p_threshold=0.05, max_depth=4):
        self.feature_names = feature_names or FEATURE_NAMES
        self.min_samples = min_samples      # minimum samples before allowing a split
        self.p_threshold = p_threshold      # t-test significance threshold
        self.max_depth = max_depth
        self.tree = None

    def fit(self, df):
        states = np.stack(df['state'].values)
        rewards = df['rewards'].values
        self.tree = self._build_tree(states, rewards, depth=0)
        return self

    def _best_split(self, states, rewards):
        """Find the feature whose split produces the most significant reward difference."""
        best_feature, best_p, best_t = None, 1.0, 0.0

        for feat_idx in range(states.shape[1]):
            values = np.unique(states[:, feat_idx])
            if len(values) < 2:
                continue
            # try every midpoint as a split threshold
            for threshold in (values[:-1] + values[1:]) / 2:
                mask = states[:, feat_idx] <= threshold
                left_rewards, right_rewards = rewards[mask], rewards[~mask]
                if len(left_rewards) < self.min_samples or len(right_rewards) < self.min_samples:
                    continue
                t_stat, p_val = stats.ttest_ind(left_rewards, right_rewards)
                if p_val < best_p:
                    best_p = p_val
                    best_t = t_stat
                    best_feature = (feat_idx, threshold)

        return best_feature, best_p, best_t

    def _build_tree(self, states, rewards, depth):
        node = {
            'n_samples': len(rewards),
            'mean_reward': np.mean(rewards),
            'std_reward': np.std(rewards),
            'depth': depth,
            'feature': None,
            'children': None
        }

        if depth >= self.max_depth or len(rewards) < 2 * self.min_samples:
            return node

        (best_feature, threshold), best_p, _ = self._best_split(states, rewards) if self._best_split(states, rewards)[0] else ((None, None), 1.0, 0.0)

        # only split if statistically significant
        if best_feature is None or best_p >= self.p_threshold:
            return node

        mask = states[:, best_feature] <= threshold
        node['feature'] = best_feature
        node['feature_name'] = self.feature_names[best_feature]
        node['threshold'] = threshold
        node['p_value'] = best_p
        node['children'] = {
            f'<= {threshold:.2f}': self._build_tree(states[mask],  rewards[mask],  depth + 1),
            f'>  {threshold:.2f}': self._build_tree(states[~mask], rewards[~mask], depth + 1)
        }
        return node

    def relevant_features(self):
        """Return features that were split on at any point in the tree."""
        features = {}
        self._collect_features(self.tree, features)
        return sorted(features.items(), key=lambda x: x[1]['min_p'])

    def _collect_features(self, node, features):
        if node['feature'] is None:
            return
        name = node['feature_name']
        if name not in features:
            features[name] = {'count': 0, 'min_p': 1.0}
        features[name]['count'] += 1
        features[name]['min_p'] = min(features[name]['min_p'], node['p_value'])
        for child in node['children'].values():
            self._collect_features(child, features)

    def print_tree(self, node=None, indent=0):
        node = node or self.tree
        prefix = '  ' * indent
        if node['feature'] is None:
            print(f"{prefix}Leaf: mean_reward={node['mean_reward']:.3f}, n={node['n_samples']}")
        else:
            print(f"{prefix}Split on '{node['feature_name']}' <= {node['threshold']:.2f} "
                  f"(p={node['p_value']:.4f}, n={node['n_samples']})")
            for label, child in node['children'].items():
                print(f"{prefix}  [{label}]")
                self.print_tree(child, indent + 2)

# ── usage ────────────────────────────────────────────────────────────────────

# Random forest importance
importance_df = rf_feature_importance(simulation_results)
print(importance_df)

# G-algorithm
g = GAlgorithm(feature_names=FEATURE_NAMES, min_samples=30, p_threshold=0.05, max_depth=4)
g.fit(simulation_results)
g.print_tree()

print("\nRelevant features found by G-algorithm:")
for feat, info in g.relevant_features():
    print(f"  {feat}: split {info['count']} time(s), min p-value={info['min_p']:.4f}")

In [11]:
X = df[state_cols + ["action_id"]]
y = df[next_state_cols]

model_transition = RandomForestRegressor()
model_transition.fit(X, y)

result = permutation_importance(model_transition, X, y, n_repeats=10)

importance = pd.Series(result.importances_mean, index=X.columns)
print(importance.sort_values(ascending=False))

s3           0.694062
s2           0.663488
s4           0.419744
action_id    0.351323
s0           0.189516
s1           0.187202
dtype: float64


In [12]:
reward_cols = ["r_expert", "obs_time", "obs_liked", "obs_pu", "obs_diff"]

importances = {}

for col in reward_cols:
    model = RandomForestRegressor()
    model.fit(X, df[col])
    
    result = permutation_importance(model, X, df[col], n_repeats=10)
    importances[col] = result.importances_mean

importance_df = pd.DataFrame(importances, index=X.columns)
print(importance_df)

           r_expert  obs_time  obs_liked    obs_pu  obs_diff
s0         0.000000  0.303683   0.366617  0.380853  0.314134
s1         0.000000  0.350535   0.434345  0.495803  0.357993
s2         0.000000  0.311764   0.397468  0.571654  0.355328
s3         0.000000  0.368114   0.437160  0.464358  0.391110
s4         0.000000  0.213678   0.240695  0.224824  0.211087
action_id  2.007945  1.051549   0.901267  0.758091  1.018229
